In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig
import pickle
import warnings
import os
import gc

warnings.filterwarnings('ignore')

In [ ]:
class Config:
    MAX_LENGTH = 256
    BATCH_SIZE = 2
    LEARNING_RATE = 3e-5
    EPOCHS = 5
    FOLDS = 5
    DROPOUT = 0.1
    HIDDEN_DIM = 384
    GRAD_ACCUMULATION_STEPS = 16
    MODEL_NAME = 'distilbert-base-uncased'
    FREEZE_LAYERS = 4
    LABEL_SMOOTHING = 0.05
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01

In [ ]:
class LightweightChatbotModel(nn.Module):
    def __init__(self, model_config_dir_path, dropout_rate, freeze_layers_count=0):
        super().__init__()
        print(f"Initializing backbone using local config from directory: {model_config_dir_path}")

        try:
            backbone_config = AutoConfig.from_pretrained(model_config_dir_path, local_files_only=True)
        except Exception as e:
            print(f"Failed to load AutoConfig from {model_config_dir_path}. Ensure config.json exists and is valid.")
            raise e

        print(f"Initializing AutoModel from loaded config (model type: {backbone_config.model_type}). Weights will be random at this stage.")
        self.backbone = AutoModel.from_config(backbone_config) 
        
        backbone_hidden_size = self.backbone.config.hidden_size
        
        if freeze_layers_count > 0:
            layers_to_freeze_attr = None
            if hasattr(self.backbone, 'transformer') and hasattr(self.backbone.transformer, 'layer'):
                layers_to_freeze_attr = self.backbone.transformer.layer
            elif hasattr(self.backbone, 'encoder') and hasattr(self.backbone.encoder, 'layer'):
                layers_to_freeze_attr = self.backbone.encoder.layer
            if layers_to_freeze_attr:
                num_to_freeze = min(freeze_layers_count, len(layers_to_freeze_attr))
                for i, layer in enumerate(layers_to_freeze_attr):
                    if i < num_to_freeze:
                        for param in layer.parameters():
                            param.requires_grad = False
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Sequential(
            nn.Linear(backbone_hidden_size, backbone_hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(backbone_hidden_size // 2, 3)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids, attention_mask, output_hidden_states=False)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

In [ ]:
class OptimizedInferenceDataset(Dataset):
    def __init__(self, data_dict, tokenizer, max_length=512):
        self.prompts = data_dict['prompts']
        self.responses_a = data_dict['responses_a'] 
        self.responses_b = data_dict['responses_b']
        self.targets = data_dict.get('targets', None)
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.prompts)
    
    def __getitem__(self, idx):
        prompt = str(self.prompts[idx])
        response_a = str(self.responses_a[idx])
        response_b = str(self.responses_b[idx])
        
        text = f"{prompt} {self.tokenizer.sep_token} {response_a} {self.tokenizer.sep_token} {response_b}"
        
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors="pt",
            add_special_tokens=True
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }

In [ ]:
class EnsembleModel:
    def __init__(self, model_paths, tokenizer, device, loaded_config, backbone_config_dir):
        self.models = []
        self.tokenizer = tokenizer
        self.device = device
        self.loaded_config = loaded_config # Store loaded_config
        self.backbone_config_dir = backbone_config_dir

        for path in model_paths:
            if os.path.exists(path):
                model = LightweightChatbotModel(
                    model_config_dir_path=self.backbone_config_dir,
                    dropout_rate=self.loaded_config.DROPOUT,
                    freeze_layers_count=self.loaded_config.FREEZE_LAYERS
                )
                # Muat state_dict secara langsung
                model.load_state_dict(torch.load(path, map_location=device))
                model.to(device)
                model.eval()
                self.models.append(model)
                print(f"Loaded model from {path}")
            else:
                print(f"WARNING: Model path not found: {path}")
        
        if not self.models:
            print("ERROR: No models were loaded into the ensemble. Check paths and model files.")

    def predict(self, data_dict, batch_size=8):
        dataset = OptimizedInferenceDataset(data_dict, self.tokenizer, self.loaded_config.MAX_LENGTH)
        
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
        
        all_preds_ensemble = []
        
        if not self.models:
            print("Error: Cannot predict, no models in ensemble.")
            return np.array([])

        with torch.no_grad():
            for batch_idx, batch in enumerate(loader): # Tambahkan batch_idx untuk logging
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                
                batch_model_probs = []
                for model_instance in self.models:
                    try:
                        if self.device.type == 'cuda':
                            with torch.cuda.amp.autocast(enabled=True):
                                logits = model_instance(input_ids, attention_mask)
                                probs = torch.softmax(logits, dim=1)
                        else:
                            logits = model_instance(input_ids, attention_mask)
                            probs = torch.softmax(logits, dim=1)
                        batch_model_probs.append(probs.cpu().numpy())
                    except Exception as e:
                        print(f"Error during prediction with one of the ensemble models: {e}")
                        continue
                
                if batch_model_probs:
                    avg_batch_probs = np.mean(batch_model_probs, axis=0)
                    all_preds_ensemble.extend(avg_batch_probs)
                else:
                    print(f"Warning: No predictions generated for batch {batch_idx} from any ensemble model.")


                if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == len(loader):
                    print(f"Ensemble processed batch {batch_idx + 1}/{len(loader)}")
                    if self.device.type == 'cuda': clear_memory()
        
        return np.array(all_preds_ensemble)

In [ ]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def main_inference_offline():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    MODEL_ARTIFACTS_BASE_PATH = "/kaggle/input/llm-classification-finetuning-version-3/pytorch/default/5"
    
    CUSTOM_CONFIG_PKL_PATH = os.path.join(MODEL_ARTIFACTS_BASE_PATH, "model_config.pkl")
    TOKENIZER_DIR_PATH = os.path.join(MODEL_ARTIFACTS_BASE_PATH, "tokenizer")
    NUM_FOLDS = 5
    MODEL_WEIGHTS_PATHS = [
        os.path.join(MODEL_ARTIFACTS_BASE_PATH, f"best_model_fold_{i}.pth") for i in range(NUM_FOLDS)
    ]
    BACKBONE_CONFIG_DIR_PATH = MODEL_ARTIFACTS_BASE_PATH 
    
    TEST_DATA_PATH = "/kaggle/input/llm-classification-finetuning/test.csv"
    SUBMISSION_FILE_PATH = "/kaggle/working/submission.csv"

    print("Loading custom model configuration (Config class)...")
    try:
        with open(CUSTOM_CONFIG_PKL_PATH, 'rb') as f:
            loaded_custom_config = pickle.load(f)
        print(f"Custom config loaded: MAX_LENGTH={loaded_custom_config.MAX_LENGTH}, BATCH_SIZE={loaded_custom_config.BATCH_SIZE}, DROPOUT={loaded_custom_config.DROPOUT}, FOLDS={getattr(loaded_custom_config, 'FOLDS', 'N/A')}")
        # Update NUM_FOLDS jika ada di config, jika tidak, gunakan nilai default di atas
        if hasattr(loaded_custom_config, 'FOLDS'):
            NUM_FOLDS = loaded_custom_config.FOLDS
            MODEL_WEIGHTS_PATHS = [
                os.path.join(MODEL_ARTIFACTS_BASE_PATH, f"best_model_fold_{i}.pth") for i in range(NUM_FOLDS)
            ]

    except FileNotFoundError:
        print(f"ERROR: Custom config file not found at {CUSTOM_CONFIG_PKL_PATH}.")
        return
    except Exception as e:
        print(f"ERROR loading custom config: {e}")
        return

    print("Loading tokenizer locally...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR_PATH, local_files_only=True)
        print(f"Tokenizer loaded locally from {TOKENIZER_DIR_PATH}")
    except Exception as e:
        print(f"ERROR loading tokenizer locally from {TOKENIZER_DIR_PATH}: {e}")
        return

    print("Initializing Ensemble Model...")
    try:
        ensemble_model = EnsembleModel(
            model_paths=MODEL_WEIGHTS_PATHS,
            tokenizer=tokenizer,
            device=device,
            loaded_config=loaded_custom_config,
            backbone_config_dir=BACKBONE_CONFIG_DIR_PATH
        )
        if not ensemble_model.models:
             print("ERROR: Ensemble model initialization failed or no models were loaded.")
             return
        print(f"Ensemble model initialized with {len(ensemble_model.models)} models.")

    except Exception as e:
        print(f"ERROR initializing or loading ensemble model: {e}")
        return

    print("Loading and preprocessing test data...")
    try:
        test_df = pd.read_csv(TEST_DATA_PATH)
        test_df['prompt'] = test_df['prompt'].fillna('').astype(str) # .str[:200]
        test_df['response_a'] = test_df['response_a'].fillna('').astype(str) # .str[:300]
        test_df['response_b'] = test_df['response_b'].fillna('').astype(str) # .str[:300]
    except FileNotFoundError:
        print(f"ERROR: Test data file not found at {TEST_DATA_PATH}.")
        return
    
    test_data_dict = {
        'prompts': test_df['prompt'].values,
        'responses_a': test_df['response_a'].values,
        'responses_b': test_df['response_b'].values
    }

    inference_batch_size_for_ensemble = loaded_custom_config.BATCH_SIZE * 8 
    print(f"Using inference batch size for ensemble: {inference_batch_size_for_ensemble}")

    print("Generating predictions with Ensemble Model...")
    predictions_array = ensemble_model.predict(test_data_dict, batch_size=inference_batch_size_for_ensemble)

    if predictions_array.size == 0:
        print("No predictions were generated by the ensemble. Check for errors in model loading or prediction loop.")
        return

    if predictions_array.ndim != 2 or predictions_array.shape[1] != 3:
        print(f"ERROR: Predictions array has unexpected shape: {predictions_array.shape}. Expected (n_samples, 3).")
        return
    if len(test_df) != predictions_array.shape[0]:
        print(f"ERROR: Number of predictions ({predictions_array.shape[0]}) does not match number of test samples ({len(test_df)}).")
        return


    submission_df = pd.DataFrame({'id': test_df['id']})
    submission_df['winner_model_a'] = predictions_array[:, 0]
    submission_df['winner_model_b'] = predictions_array[:, 1]
    submission_df['winner_tie'] = predictions_array[:, 2]
    
    prob_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
   
    row_sums = submission_df[prob_cols].sum(axis=1)
   
    safe_row_sums = np.where(row_sums == 0, 1e-9, row_sums) 
    submission_df[prob_cols] = submission_df[prob_cols].div(safe_row_sums, axis=0)
    
    submission_df.to_csv(SUBMISSION_FILE_PATH, index=False)
    print(f"Submission file created: {SUBMISSION_FILE_PATH}")
    print(f"Submission shape: {submission_df.shape}")
    print("Submission head:\n", submission_df.head())
    
    final_sums_check = submission_df[prob_cols].sum(axis=1)
    print(f"\nFinal probability sums check: min={final_sums_check.min():.6f}, max={final_sums_check.max():.6f}, mean={final_sums_check.mean():.6f}")
    if submission_df[prob_cols].isnull().any().any():
        print("WARNING: NaNs found in submission probabilities!")
    else:
        print("No NaNs found in submission probabilities.")

    if device.type == 'cuda': clear_memory()

if __name__ == "__main__":
    main_inference_offline()